# SRA Taxonomy Analysis Tool (STAT)

This notebook demonstrates how to use [NCBI STAT]((https://www.ncbi.nlm.nih.gov/sra/docs/sra-taxonomy-analysis-tool/)) for **taxonomy-derived information** from a small sequence input.

**What is STAT?**

The SRA Taxonomy Analysis Tool (STAT) is a collection of tools for building k-mer databases, and querying against them. It maps individual sequencing reads to a taxonomic hierarchy and reports the taxonomic composition of reads within a sequencing run.

Input your sequencing reads, and STAT will try to match those reads to known organisms in NCBI’s taxonomy database. The output helps you answer questions like:

- “Is this sample mostly human?”
- “Does it contain bacteria, viruses, fungi, or parasites?”
- “Are there unexpected organisms in this SRA run?”
- “Which runs are worth downloading and studying further?”

So, STAT is used to quickly check what organisms appear to be present in sequencing data. It does this with exact read matches against precomputed **k-mer dictionary databases**, using a coarse first pass to identify likely organisms and a finer second pass to assign reads across species and higher-level taxonomy groups.


| STAT command or script | What it does in this demo |
|---|---|
| `quickbuild.sh` | Builds the minimal STAT tools needed to run the bundled example. |
| `build_index_of_each_file` | Builds per-file k-mer indexes from the example FASTA database files. |
| `merge_db` | Merges the per-file indexes into one combined example database. |
| `identify_tax_ids` | Links database k-mers to taxonomy IDs using the `tax.parents` file. |
| `db_tax_id_to_dbs` | Combines the database and taxonomy-ID mapping into a `.dbs` file. |
| `sort_dbs` | Sorts the dense `.dbs` file by tax ID for optional two-step workflows. |
| `aligns_to` | Queries the input FASTA file against the STAT database and writes the `.hits` output file. |


**Note:** Commands that begin with `!` run shell commands inside the notebook. Cells that begin with `%%bash` run the whole cell as Bash.

---

## 1. Set up the workspace

In this code block, we will downloads the `tax` branch of `ncbi/ngs-tools` into `~/stat_demo_workspace/ngs-tools-stat` and store the STAT tool path in `STAT_DIR`.

Using a fixed workspace avoids the rerun bug where the repo was cloned inside a previous `tools/tax` folder.

In [ ]:
# Clone STAT into a fixed workspace so rerunning the notebook does not nest repos.

from pathlib import Path
import os
import shutil

REPO_URL = "https://github.com/ncbi/ngs-tools.git"
REPO_BRANCH = "tax"
DEMO_ROOT = Path.home() / "stat_demo_workspace"
REPO_DIR = DEMO_ROOT / "ngs-tools-stat"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

DEMO_ROOT.mkdir(parents=True, exist_ok=True)

!git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

STAT_DIR = REPO_DIR / "tools" / "tax"
os.environ["STAT_DIR"] = str(STAT_DIR)

print("STAT directory:", STAT_DIR)

print("\n\n-------------\nTASK COMPLETE\n-------------")

## 2. Review STAT quickstart file

Inspect the README, bundled example FASTA inputs, and the key commands in the official STAT example script before running anything.

**NOTE:** This isn't necessary to set up an analysis, but it should be a good place to start when it comes time to integrate STAT into your specific pipeline.

In [ ]:
# Inspect the STAT quick start and the bundled example.

!sed -n '1,40p' "$STAT_DIR/README.md"

print("\n--- Example input files ---")
!find "$STAT_DIR/examples/example_data" -maxdepth 1 -type f | sort | sed -n '1,20p'

print("\n--- Key commands in the official example ---")
!grep -nEi 'DENSE_WINDOW|SPARSE_WINDOW|KMER_LEN|build_index|merge_db|identify_tax_ids|db_tax_id_to_dbs|sort_dbs|aligns_to|hits_to_report|python2' "$STAT_DIR/examples/build_db_and_run.sh" | sed -n '1,100p'

print("\n\n-------------\nTASK COMPLETE\n-------------")

## 3. Build the minimal STAT tools

Next we will run STAT's `quickbuild.sh` minimal build script and confirm that the command-line tools have been installed.

If the build somehow fails, then you should check that your current environment has Git, Python 3, a C++ compiler, and OpenMP support.

In [ ]:
%%bash
# Build the minimal STAT tools needed for the example.

echo "Running STAT's minimal build script."
echo "Please be patient. This might take a moment..."

set -euo pipefail
cd "$STAT_DIR"

bash ./quickbuild.sh

echo
echo "--- Tools used in this demo ---"
for tool in aligns_to build_index_of_each_file merge_db identify_tax_ids db_tax_id_to_dbs sort_dbs; do
    if [ -x "./bin/$tool" ]; then
        echo "./bin/$tool"
    elif [ -e "./bin/$tool" ]; then
        echo "./bin/$tool found, but not marked executable"
    else
        echo "./bin/$tool not found"
    fi
done

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

## 4. Align a FASTA to a k-mer database

In this section, we will first build a small demo k-mer database from the bundled example reference files, and then use the `aligns_to` command to query one bundled FASTA file against that database.

`aligns_to` is one of the main STAT query commands you will want to use. It compares reads from a FASTA (in this case, `SRR4841604.fasta`) against a STAT database, writing the matching taxonomy-derived hits to a`*.hits` file (here, it is `SRR4841604.fasta.hits`).

The notebook stops after this first query so the demo stays short and avoids later optional reporting steps. For additional workflows, including two-step processing and report generation, see the official [STAT GitHub repository](https://github.com/ncbi/ngs-tools/tree/tax/tools/tax).

In [ ]:
%%bash
# Run a shortened version of the official STAT example.
# Builds the small example database and runs the first aligns_to query only.

set -euo pipefail
cd "$STAT_DIR/examples"

# Clean old demo outputs if this notebook is rerun.
rm -f files.list.example example.* SRR4841604.fasta*.hits *.log build_db_and_run_demo.sh tax_list taxdump.tar.gz
find sequence_tree \( -name '*.dense.db' -o -name '*.sparse.db' -o -name '*.summary.tsv' \) -type f -exec rm -f {} +

# Copy the official example, through to the first one-step aligns_to command.
awk '{ print } /\$bin_dir\/aligns_to -dbs .*SRR4841604\.fasta/ { exit }' build_db_and_run.sh > build_db_and_run_demo.sh

if ! grep -q 'aligns_to' build_db_and_run_demo.sh; then
    echo "Could not find the one-step aligns_to command in build_db_and_run.sh."
    exit 1
fi

echo "--- STAT query command used in this demo ---"
grep -n 'aligns_to' build_db_and_run_demo.sh

if ! bash ./build_db_and_run_demo.sh > stat_demo_run.log 2>&1; then
    echo
    echo "--- Demo run failed; last 80 log lines ---"
    tail -80 stat_demo_run.log
    exit 1
fi

echo
echo "--- Selected run log lines ---"
grep -Ei 'version|window size|kmer len|kmers|FastaReader|total spot count|total read count|total time' stat_demo_run.log | sed -n '1,80p'

test -s SRR4841604.fasta.hits

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

## 5. Preview the taxonomy-derived output

At this point, we should have successfully created our `.hits` output file.

Each row starts with an input read/spot identifier followed by a taxon-hit summary such as `211044x17`; in this example, `211044` is a taxonomic identifier and `17` is the associated hit count for that record.

**NOTE:** In this shortened demo, the `.hits` file is the main output we will inspect. In a fuller STAT workflow, this `.hit` file would usually be treated as an intermediate result that can be summarized into a more readable taxonomy report.

In [ ]:
%%bash
# Preview only useful text outputs from the demo.

set -euo pipefail
cd "$STAT_DIR/examples"

echo "--- Main STAT output files ---"
ls -lh SRR4841604.fasta.hits stat_demo_run.log

echo
echo "--- First lines of the STAT hit output ---"
sed -n '1,12p' SRR4841604.fasta.hits

echo
echo "--- A few index summary tables created during database build ---"
find sequence_tree -name '*.summary.tsv' -type f | sort | head -n 3 | while read -r f; do
    echo
    echo "===== $f ====="
    sed -n '1,5p' "$f"
done

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"


## 6. (Optional) Create a full STAT report

As previously described, this demo stops at creating a `.hits` file because the official reporting helpers in this STAT example expect a `python2` command to be available. This requires additional steps which would complicate this quickstart Jupyter demo.

That said, if you want to set up a `python2` environment for your own workflow, and have it installed and available on your `PATH`, then you can copy/paste one of the blocks below after the database has been built.

This follows the later part of the official example: first query the sparse database, convert those first-pass hits into a `tax_list`, then query the sorted dense database using that tax list.

In [ ]:
%%bash
# (Optional) Activate Python 2 environment.
# Uncomment the below two lines if you installed Python 2 in a conda environment.
#source "$(conda info --base)/etc/profile.d/conda.sh"
#conda activate stat-py2

set -euo pipefail

cd "$STAT_DIR/examples"

bin_dir="../bin"

# Step 1: Run a broad first pass against the sparse database.
$bin_dir/aligns_to -dbs ./example.sparse.dbs ./example_data/SRR4841604.fasta > ./SRR4841604.fasta.1ststep.hits

# Step 2: Convert first-pass hits into a tax ID list.
python2 $bin_dir/hits_to_tax_list.py ./SRR4841604.fasta.1ststep.hits > ./tax_list

# Step 3: Run a focused second pass against the dense database,
# using only the tax IDs found in the first pass.
$bin_dir/aligns_to -dbss ./example.dense.dbss -tax_list ./tax_list ./example_data/SRR4841604.fasta > ./SRR4841604.fasta.2ndstep.hits

# Step 4: Generate a readable report from the second-pass hits.
$bin_dir/hits_to_report.sh ./SRR4841604.fasta.2ndstep.hits

printf "\n\n-------------\nTASK COMPLETE\n-------------\n"

⚠️ **IMPORTANT!** If you try to run the above codeblock in this notebook, it will fail. This is expected!

---

If you have any more specific questions, please visit the [STAT webpage](https://www.ncbi.nlm.nih.gov/sra/docs/sra-taxonomy-analysis-tool/), the [STAT GitHub repository](https://github.com/ncbi/ngs-tools/tree/tax/tools/tax), or email your questions to [sra@ncbi.nlm.nih.gov](mailto:sra@ncbi.nlm.nih.gov).
